# Deep Learning CIA (2087101): Fashion-MNIST Autoencoders

**Experiments**
1. Train an undercomplete autoencoder and evaluate reconstruction MSE and theoretical compression ratio.
2. Compare latent dimensions 8, 32, and 64.
3. Train a denoising autoencoder that reconstructs clean images from noisy inputs.
4. Plot training curves and display original, noisy, and reconstructed images.

**Framework:** PyTorch  
**Dataset:** Fashion-MNIST (28 × 28 grayscale images)  
**Reproducibility:** Fixed random seeds. Run all cells from top to bottom.

> Dataset note: the first run may download Fashion-MNIST and therefore requires an internet connection. Metrics in the report should be copied from the actual notebook output after running it.

## 1. Setup and dataset

Fashion-MNIST contains 28 × 28 grayscale clothing images. Each image has 784 pixel values. The images are scaled to [0, 1] and flattened to vectors of length 784 for a fully connected autoencoder.

In [ ]:
# If needed, install dependencies in your environment:
# pip install torch torchvision matplotlib numpy

import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

transform = transforms.ToTensor()  # pixel values become floats in [0, 1]
train_ds = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_ds = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

# Flatten each 1×28×28 image to 784 features.
X_train = train_ds.data.float().view(-1, 784) / 255.0
X_test = test_ds.data.float().view(-1, 784) / 255.0

# Use a smaller subset for quicker CIA experimentation; change to all rows for a full run.
TRAIN_LIMIT = 12000
TEST_LIMIT = 3000
X_train = X_train[:TRAIN_LIMIT]
X_test = X_test[:TEST_LIMIT]

train_loader = DataLoader(TensorDataset(X_train), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test), batch_size=256, shuffle=False)
print("Train images:", len(X_train), "| Test images:", len(X_test), "| Input dimensions:", X_train.shape[1])

## 2. Undercomplete autoencoder architecture

The encoder compresses 784 input values into a small latent vector. The decoder maps the latent vector back to 784 output values. A sigmoid output keeps reconstructed pixels in [0, 1].

**Architecture:** `784 → 256 → 128 → latent dimension → 128 → 256 → 784`

The bottleneck is undercomplete because its dimension is much smaller than 784.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        return reconstructed

def evaluate_mse(model, loader):
    model.eval()
    total_squared_error, total_elements = 0.0, 0
    with torch.no_grad():
        for (images,) in loader:
            images = images.to(device)
            predictions = model(images)
            total_squared_error += torch.sum((predictions - images) ** 2).item()
            total_elements += images.numel()
    return total_squared_error / total_elements

def train_autoencoder(latent_dim=32, epochs=10, noise_std=0.0, weight_decay=0.0):
    model = Autoencoder(latent_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    criterion = nn.MSELoss()
    history = []

    for epoch in range(epochs):
        model.train()
        total_loss, count = 0.0, 0
        for (clean,) in train_loader:
            clean = clean.to(device)
            # When noise_std > 0, the input is corrupted but target remains clean.
            if noise_std > 0:
                corrupted = torch.clamp(clean + noise_std * torch.randn_like(clean), 0.0, 1.0)
            else:
                corrupted = clean
            optimizer.zero_grad()
            reconstructed = model(corrupted)
            loss = criterion(reconstructed, clean)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * clean.size(0)
            count += clean.size(0)
        epoch_loss = total_loss / count
        history.append(epoch_loss)
        print(f"Latent={latent_dim:>2} | Epoch {epoch+1:02d}/{epochs} | Train MSE={epoch_loss:.6f}")

    return model, history

## 3. Experiment 1: Train an undercomplete autoencoder

This experiment uses a latent size of 32 as the baseline. MSE is calculated over all test pixels. The simple dimensional compression ratio is input dimensions divided by latent dimensions; this ratio does not include model weights, metadata, or storage overhead.

In [ ]:
EPOCHS = 10
baseline_dim = 32
baseline_model, baseline_history = train_autoencoder(latent_dim=baseline_dim, epochs=EPOCHS)

baseline_test_mse = evaluate_mse(baseline_model, test_loader)
input_dim = 784
compression_ratio = input_dim / baseline_dim

print(f"Baseline latent dimension: {baseline_dim}")
print(f"Test reconstruction MSE: {baseline_test_mse:.6f}")
print(f"Dimensional compression ratio: {input_dim}:{baseline_dim} = {compression_ratio:.2f}:1")

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(baseline_history)+1), baseline_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training MSE")
plt.title("Undercomplete Autoencoder: Training Loss (latent=32)")
plt.grid(True)
plt.tight_layout()
plt.show()

## 4. Experiment 2: Compare latent dimensions 8, 32, and 64

All models use the same architecture apart from the bottleneck size and are trained for the same number of epochs. Lower MSE indicates better pixel-level reconstruction. Larger latent dimensions generally preserve more information but have a lower dimensional compression ratio.

In [ ]:
latent_sizes = [8, 32, 64]
comparison_models = {}
comparison_histories = {}
comparison_results = []

for latent_dim in latent_sizes:
    model, history = train_autoencoder(latent_dim=latent_dim, epochs=EPOCHS)
    test_mse = evaluate_mse(model, test_loader)
    ratio = 784 / latent_dim
    comparison_models[latent_dim] = model
    comparison_histories[latent_dim] = history
    comparison_results.append((latent_dim, test_mse, ratio))
    print(f"Latent={latent_dim:>2} | Test MSE={test_mse:.6f} | Dimensional compression ratio={ratio:.2f}:1")

print("\nSummary (copy these actual values into your report):")
print("Latent dimension | Test MSE | Compression ratio")
for latent_dim, test_mse, ratio in comparison_results:
    print(f"{latent_dim:>15} | {test_mse:.6f} | {ratio:.2f}:1")

plt.figure(figsize=(8, 4))
for latent_dim, history in comparison_histories.items():
    plt.plot(range(1, len(history)+1), history, marker=".", label=f"Latent {latent_dim}")
plt.xlabel("Epoch")
plt.ylabel("Training MSE")
plt.title("Training Loss by Bottleneck Dimension")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 5. Denoising autoencoder

A denoising autoencoder receives noisy images as inputs but uses the original clean images as targets. The model learns to remove some of the added Gaussian noise while reconstructing the image.

In [ ]:
noise_std = 0.35
denoising_model, denoising_history = train_autoencoder(
    latent_dim=32, epochs=EPOCHS, noise_std=noise_std
)

# Evaluate denoising on noisy test inputs, with clean images as targets.
denoising_model.eval()
total_error, total_elements = 0.0, 0
with torch.no_grad():
    for (clean,) in test_loader:
        clean = clean.to(device)
        noisy = torch.clamp(clean + noise_std * torch.randn_like(clean), 0.0, 1.0)
        reconstructed = denoising_model(noisy)
        total_error += torch.sum((reconstructed - clean) ** 2).item()
        total_elements += clean.numel()
denoising_test_mse = total_error / total_elements
print(f"Denoising test MSE (noisy input vs clean target): {denoising_test_mse:.6f}")

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(denoising_history)+1), denoising_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("MSE against clean target")
plt.title("Denoising Autoencoder Training Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

## 6. Visualize original, noisy, and reconstructed images

The first row shows clean images, the second row shows their corrupted inputs, and the third row shows denoised reconstructions.

In [ ]:
denoising_model.eval()
images = X_test[:8].to(device)
torch.manual_seed(2026)
noisy_images = torch.clamp(images + noise_std * torch.randn_like(images), 0.0, 1.0)
with torch.no_grad():
    reconstructions = denoising_model(noisy_images)

fig, axes = plt.subplots(3, 8, figsize=(14, 5))
row_titles = ["Original", "Noisy input", "Reconstruction"]
for col in range(8):
    axes[0, col].imshow(images[col].cpu().reshape(28, 28), cmap="gray")
    axes[1, col].imshow(noisy_images[col].cpu().reshape(28, 28), cmap="gray")
    axes[2, col].imshow(reconstructions[col].cpu().reshape(28, 28), cmap="gray")
    for row in range(3):
        axes[row, col].axis("off")
for row, title in enumerate(row_titles):
    axes[row, 0].set_ylabel(title, rotation=90, size=10)
plt.suptitle("Fashion-MNIST Denoising Autoencoder")
plt.tight_layout()
plt.show()

## 7. Analysis and discussion

Complete this section using the values printed by your run.

- **Latent size 8:** highest dimensional compression (784/8 = 98:1), but it may discard fine details and produce higher reconstruction MSE.
- **Latent size 32:** middle-ground compression (24.5:1) and reconstruction capacity.
- **Latent size 64:** lower dimensional compression (12.25:1) but more capacity to preserve image detail.
- **Denoising:** the model is trained with noisy inputs and clean targets. Compare its visual outputs and noisy-input test MSE with the baseline. Do not directly compare losses from different input conditions without explaining the difference.
- **Regularization:** this notebook exposes `weight_decay` in `train_autoencoder`. To test L2 regularization, rerun a model with a nonzero value such as `weight_decay=1e-4` and compare its test MSE. Keep the training setup otherwise consistent.

**Compression ratio definition:** this notebook reports *dimensional* compression ratio, `784 / latent_dim`. It is not a complete file-size compression ratio because model parameters and encoding/storage overhead are excluded.

## Conclusion

An undercomplete autoencoder learns a compact latent representation of Fashion-MNIST images and reconstructs them by minimizing MSE. Comparing bottleneck sizes illustrates the trade-off between compression and reconstruction fidelity. A denoising autoencoder is trained with corrupted inputs and clean targets to learn noise-robust representations. Report the actual measured MSE values and visual observations from your run; results vary with epochs, data subset, hardware, and random seed.

## Submission checklist
- [ ] Run all cells from top to bottom.
- [ ] Confirm training logs, plots, comparison metrics, and image grids are visible.
- [ ] Fill the report table using actual printed results.
- [ ] Add name, roll number, class, and date if required.
- [ ] Upload the `.ipynb`, `README.md`, `output_logs.txt`, and PDF report to GitHub.